# Best fit value of the normalisation parameter, a, in the dust MBB

This notebook is to determine what value of the normalisation constant in the equivalent dust temperature equation best matches the observed data from Simon Driver. First the best match for the peaks in the far IR regime will be found and then the best fit for the overall data will also be determined. Pipeline is going to first run for m25n256 and then m100n1024 although big differences in results are not expected as only volume size is changing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys 

import h5py
import caesar
sys.path.append("/home/spujni/simba_cosmic_background")

from src.utils import load_background_results, load_farIR_parameter_sweep


In [ ]:
#running here with m25n256 but needs to be done for m100n1024 for any actual results because ebl data is from that i think.
df = pd.read_csv(r"/home/spujni/simba_cosmic_background/data/ebl/ebldata.csv")
df = df.sort_values(by = "wave")

# df = df[~(((df["wave"] == 60) | (df["wave"] == 100)) & (df["paper"] == "casandjian2024"))]
#df = df[~(((df["wave"] == 100)) & (df["paper"] == "casandjian2024"))]

obs_lam = df["wave"] * 1e4  # convert µm → AA
obs_nuInu = df["ebl"]
obs_nuInu_err = df["debl"]

data = load_farIR_parameter_sweep("m100n1024", z_min=0.0, z_max=7.0) # m100n1024 run for multiple different values of a


# Chi-squared minimisation 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# ── Wavelength window (microns) ──
LAM_MIN, LAM_MAX = 30.0, 3000.0

# ── Observed data in the window ──
mask = (df["wave"] >= LAM_MIN) & (df["wave"] <= LAM_MAX)
obs_lam_um = df.loc[mask, "wave"].to_numpy()
obs_flux   = df.loc[mask, "ebl"].to_numpy()
obs_err    = df.loc[mask, "debl"].to_numpy()

# ── Sweep over a values ──
a_vals = np.sort(np.asarray(data["a_values"]))
chi2   = np.zeros(len(a_vals))

for i, a in enumerate(a_vals):
    sp = data["spectra"][float(a)]
    model_lam_um = sp["lam_AA"] / 1e4                       # AA → µm
    model_flux   = sp["nuInu_nW"]

    f = interp1d(model_lam_um, model_flux, kind="linear",
                 bounds_error=False, fill_value=0.0)
    model_on_obs = f(obs_lam_um)

    chi2[i] = np.sum(((model_on_obs - obs_flux) / obs_err) ** 2)

# ── Best fit ──
i_best = np.argmin(chi2)
dof = len(obs_flux) - 1
print(f"Best a = {a_vals[i_best]:.4f}")
print(f"χ²     = {chi2[i_best]:.2f}")
print(f"reduced χ² = {chi2[i_best]/dof:.2f}  (dof = {dof})")

# ── Plot χ² vs a ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(a_vals, chi2 / dof, "o-", ms=5)
axes[0].axvline(a_vals[i_best], ls="--", color="tab:red",
                label=f"best a = {a_vals[i_best]:.4f}")
axes[0].set_xlabel(r"$a_{\rm dust}$")
axes[0].set_ylabel(r"$\chi^2 / \mathrm{dof}$")
axes[0].legend()

# ── Plot best-fit spectrum vs data ──
sp = data["spectra"][float(a_vals[i_best])]
axes[1].errorbar(obs_lam_um, obs_flux, yerr=obs_err,
                 fmt="o", ms=4, color="k", capsize=2, label="EBL data")
axes[1].plot(sp["lam_AA"] / 1e4, sp["nuInu_nW"],
             color="tab:red", lw=1.5, label=f"model a = {a_vals[i_best]:.4f}")
axes[1].axvline(LAM_MIN, ls="--", color="grey", lw=0.8)
axes[1].axvline(LAM_MAX, ls="--", color="grey", lw=0.8)
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel("Wavelength [µm]")
axes[1].set_ylabel(r"$\nu I_\nu$ [nW m$^{-2}$ sr$^{-1}$]")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d, RegularGridInterpolator
from scipy.optimize import minimize_scalar

# ── Wavelength window (microns) ──
LAM_MIN, LAM_MAX = 50.0, 3000.0

# ── Observed data in the window ──
mask = (df["wave"] >= LAM_MIN) & (df["wave"] <= LAM_MAX)
obs_lam_um = df.loc[mask, "wave"].to_numpy()
obs_flux   = df.loc[mask, "ebl"].to_numpy()
obs_err    = df.loc[mask, "debl"].to_numpy()

# ── Build interpolator over the pre-computed grid ──
a_vals = np.sort(np.asarray(data["a_values"]))

# Stack model fluxes into a 2D array: shape (n_a, n_obs_points)
model_grid = np.zeros((len(a_vals), len(obs_lam_um)))
for i, a in enumerate(a_vals):
    sp = data["spectra"][float(a)]
    model_lam_um = sp["lam_AA"] / 1e4
    model_flux   = sp["nuInu_nW"]
    f = interp1d(model_lam_um, model_flux, kind="linear",
                 bounds_error=False, fill_value=0.0)
    model_grid[i] = f(obs_lam_um)

# For each observed wavelength, build a 1D interpolator over a
model_interps = [interp1d(a_vals, model_grid[:, j], kind="cubic")
                 for j in range(len(obs_lam_um))]

def chi2_func(a):
    """Continuous χ² as a function of a_dust."""
    model_on_obs = np.array([mi(a) for mi in model_interps])
    return np.sum(((model_on_obs - obs_flux) / obs_err) ** 2)

# ── Optimise ──
result = minimize_scalar(chi2_func,
                         bounds=(a_vals.min(), a_vals.max()),
                         method="bounded")

a_best = result.x
chi2_best = result.fun
dof = len(obs_flux) - 1

print(f"Best a  = {a_best:.6f}")
print(f"χ²      = {chi2_best:.2f}")
print(f"χ²/dof  = {chi2_best/dof:.2f}  (dof = {dof})")

# ── Confidence interval (Δχ² = 1 for 1σ) ──
from scipy.optimize import brentq

chi2_threshold = chi2_best + 1.0

# Lower bound
try:
    a_lo = brentq(lambda a: chi2_func(a) - chi2_threshold,
                  a_vals.min(), a_best)
except ValueError:
    a_lo = a_vals.min()

# Upper bound
try:
    a_hi = brentq(lambda a: chi2_func(a) - chi2_threshold,
                  a_best, a_vals.max())
except ValueError:
    a_hi = a_vals.max()

print(f"1σ interval: a = {a_best:.4f} (+{a_hi-a_best:.4f} / -{a_best-a_lo:.4f})")

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# χ² curve (dense evaluation via interpolator)
a_fine = np.linspace(a_vals.min(), a_vals.max(), 500)
chi2_fine = np.array([chi2_func(a) for a in a_fine])

axes[0].plot(a_fine, chi2_fine / dof, "-", lw=1.5)
axes[0].axvline(a_best, ls="--", color="tab:red",
                label=f"best a = {a_best:.4f}")
axes[0].axvspan(a_lo, a_hi, alpha=0.15, color="tab:red", label="1σ")
axes[0].set_xlabel(r"$a_{\rm dust}$")
axes[0].set_ylabel(r"$\chi^2 / \mathrm{dof}$")
axes[0].legend()

# Best-fit spectrum
model_best = np.array([mi(a_best) for mi in model_interps])
axes[1].errorbar(obs_lam_um, obs_flux, yerr=obs_err,
                 fmt="o", ms=4, color="k", capsize=2, label="EBL data")
axes[1].plot(obs_lam_um, model_best,
             color="tab:red", lw=1.5, label=f"model a = {a_best:.4f}")
axes[1].axvline(LAM_MIN, ls="--", color="grey", lw=0.8)
axes[1].axvline(LAM_MAX, ls="--", color="grey", lw=0.8)
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel("Wavelength [µm]")
axes[1].set_ylabel(r"$\nu I_\nu$ [nW m$^{-2}$ sr$^{-1}$]")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
data = load_background_results("m100n1024", area=0.5, z_min=0.0, z_max=7.0)

print(data["optical"].keys())

# spectra
lam_fir = data["farIR"]["lam_AA"] * 1e-4  # µm
nuInu_fir = data["farIR"]["nuInu_nW"]

lam_opt = data["optical"]["lam_AA"] * 1e-4  # µm
nuInu_opt = data["optical"]["nuInu_nW"]

nuInu_opt_nodust = data["optical"]["nuInu_nodust_nW"]

lam_radio = data["radio"]["lam_um"]  # µm
nuInu_radio = data["radio"]["nuInu_nW"]

fir_lo, fir_hi = 50, 3000  # microns
fir_mask = (lam_fir >= fir_lo) & (lam_fir <= fir_hi) & np.isfinite(nuInu_fir)

plt.figure(figsize=(10, 6))
plt.scatter(lam_fir[fir_mask], nuInu_fir[fir_mask], marker=".", label="SIMBA Far-IR")
plt.scatter(lam_opt, nuInu_opt, marker=".", label="SIMBA Optical w Dust")
plt.scatter(lam_opt, nuInu_opt_nodust, marker=".", label="SIMBA Optical w/o Dust")
plt.scatter(lam_radio, nuInu_radio, marker=".", label="SIMBA Radio")
plt.scatter(df["wave"], df["ebl"], marker=".", label="Observed EBL")
#plt.errorbar(df["wave"], df["ebl"], yerr=df["debl"], fmt="none", ecolor="gray", alpha=0.5, label="EBL Uncertainty")
plt.xlabel("Wavelength (microns)")
plt.ylabel("EBL (nW/m^2/sr)")
plt.title("m100n1024 Simulated Extragalactic Background Light (EBL) vs Wavelength")
plt.xscale("log")
plt.yscale("log")
plt.legend()
plt.show()